---

## 📦 Imports et Configuration

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Deepchecks NLP
from deepchecks.nlp import TextData
from deepchecks.nlp.suites import train_test_validation

# ML
from sklearn.model_selection import train_test_split

In [3]:
# Configuration des chemins
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'testing' else Path.cwd()
PROCESSOR_DIR = BASE_DIR / 'processors'
TESTING_DIR = BASE_DIR / 'testing'
TESTING_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("📈 DEEPCHECKS NLP - NIVEAU 2 : DRIFT ET DISTRIBUTION")
print("="*80)
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 Base: {BASE_DIR}")
print()

📈 DEEPCHECKS NLP - NIVEAU 2 : DRIFT ET DISTRIBUTION
📅 Date: 2025-12-15 01:41:06
📁 Base: e:\MLOps\mlops_election



---

## 📥 Chargement des Données

In [4]:
def load_cleaned_texts():
    """Charge les textes nettoyés"""
    texts_path = PROCESSOR_DIR / 'cleaned_texts.pkl'
    if not texts_path.exists():
        raise FileNotFoundError(f"Textes non trouvés: {texts_path}")
    
    with open(texts_path, 'rb') as f:
        data = pickle.load(f)
    
    print(f"✅ Textes chargés: {len(data['cleaned'])} textes")
    return data['cleaned'], data['labels']

In [5]:
# Charger les données
texts, labels = load_cleaned_texts()

✅ Textes chargés: 3434 textes


---

## 📝 Création des TextData pour Deepchecks NLP

In [6]:
def create_text_data(texts_list, labels_list, split_name='train'):
    """Crée un TextData Deepchecks NLP à partir de textes et labels"""
    print(f"📝 Création TextData NLP ({split_name})")
    print("-" * 80)
    
    text_data = TextData(
        raw_text=texts_list,
        label=labels_list,
        task_type='text_classification',
        name=f'{split_name}_dataset'
    )
    
    print(f"✅ TextData créé:")
    print(f"   Nombre de textes: {len(texts_list)}")
    print(f"   Distribution labels: {pd.Series(labels_list).value_counts().to_dict()}")
    print()
    
    return text_data

In [7]:
# Créer le split train/test (même split que preprocess.py)
df = pd.DataFrame({'texts': texts, 'labels': labels})
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['labels']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['labels']
)

print(f"Split effectué:")
print(f"  Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
print()

Split effectué:
  Train: 2403 (70.0%)
  Val:   515 (15.0%)
  Test:  516 (15.0%)



In [8]:
# Créer les TextData NLP
train_text_data = create_text_data(
    train_df['texts'].tolist(), 
    train_df['labels'].tolist(), 
    'train'
)
test_text_data = create_text_data(
    test_df['texts'].tolist(), 
    test_df['labels'].tolist(), 
    'test'
)

📝 Création TextData NLP (train)
--------------------------------------------------------------------------------
✅ TextData créé:
   Nombre de textes: 2403
   Distribution labels: {0: 1233, 1: 1170}

📝 Création TextData NLP (test)
--------------------------------------------------------------------------------
✅ TextData créé:
   Nombre de textes: 516
   Distribution labels: {0: 265, 1: 251}



---

## 📈 NIVEAU 2 : TRAIN-TEST DRIFT

### Suite de validation train-test Deepchecks NLP

La suite `train_test_validation()` exécute automatiquement les checks suivants :
1. **Label Drift** : Compare la distribution des labels train vs test
2. **Property Drift** : Détecte les changements dans les propriétés textuelles
3. **Text Embeddings Drift** : Mesure le drift sémantique via embeddings
4. **Train Test Samples Mix** : Vérifie qu'aucun échantillon test n'apparaît dans train

In [9]:
def run_nlp_drift_checks(train_data, test_data):
    """NIVEAU 2: Détection de drift NLP (distribution, propriétés, labels)"""
    print("\n" + "="*80)
    print("📊 NIVEAU 2: NLP TRAIN-TEST DRIFT")
    print("="*80)
    
    # Calculer les propriétés built-in pour les checks
    print("\n⏳ Calcul des propriétés textuelles...")
    train_data.calculate_builtin_properties()
    test_data.calculate_builtin_properties()
    print("✅ Propriétés calculées")
    
    # Suite de validation train-test NLP
    drift_suite = train_test_validation()
    
    print("\n🔍 Checks de Drift NLP (suite complète):")
    print("   1. Label Drift (distribution des labels)")
    print("   2. Property Drift (longueur texte, vocabulaire)")
    print("   3. Text Embeddings Drift")
    print("   4. Train Test Samples Mix")
    
    # Exécuter la suite
    print("\n⏳ Exécution des checks de drift NLP...")
    result = drift_suite.run(train_data, test_data)
    
    # Sauvegarder le rapport
    drift_report_path = TESTING_DIR / 'deepchecks_nlp_drift_report.html'
    result.save_as_html(str(drift_report_path))
    
    print(f"✅ Rapport de drift NLP sauvegardé: {drift_report_path.name}")
    
    # Résumé des résultats
    print("\n📈 Résumé Drift NLP:")
    passed = 0
    total = 0
    for check_result in result.results:
        if hasattr(check_result, 'passed_conditions'):
            total += 1
            if check_result.passed_conditions():
                passed += 1
    
    if total > 0:
        print(f"   Checks réussies: {passed}/{total}")
    else:
        print(f"   Checks exécutés: {len(result.results)}")
    
    # Statistiques de distribution
    print("\n📊 Distribution des Labels:")
    train_labels = train_data.label
    test_labels = test_data.label
    
    print("   Train:")
    print(pd.Series(train_labels).value_counts(normalize=True).to_string())
    print("   Test:")
    print(pd.Series(test_labels).value_counts(normalize=True).to_string())
    
    return result

In [10]:
# Exécuter les checks de drift NLP
drift_result = run_nlp_drift_checks(train_text_data, test_text_data)


📊 NIVEAU 2: NLP TRAIN-TEST DRIFT

⏳ Calcul des propriétés textuelles...


100%|██████████| 33/33 [00:00<00:00, 48.54it/s]

✅ Propriétés calculées

🔍 Checks de Drift NLP (suite complète):
   1. Label Drift (distribution des labels)
   2. Property Drift (longueur texte, vocabulaire)
   3. Text Embeddings Drift
   4. Train Test Samples Mix

⏳ Exécution des checks de drift NLP...


deepchecks - WARNING - Could not find model's classes, using the observed classes. In order to make sure the classes used by the model are inferred correctly, please use the model_classes argument


✅ Rapport de drift NLP sauvegardé: deepchecks_nlp_drift_report.html

📈 Résumé Drift NLP:
   Checks réussies: 3/3

📊 Distribution des Labels:
   Train:
0    0.513109
1    0.486891
   Test:
0    0.513566
1    0.486434


In [11]:
# Afficher le widget interactif du drift
drift_result

Accordion(children=(VBox(children=(HTML(value='\n<h1 id="summary_U0PI7VVVARZ0P4BJ1I08J2RIN">Train Test Validat…

---

## 📊 Analyse détaillée des propriétés

In [12]:
# Analyser les propriétés textuelles calculées
print("📊 Propriétés textuelles disponibles:")
print("-" * 80)

if hasattr(train_text_data, '_properties'):
    print("\nPropriétés Train:")
    for prop_name in train_text_data._properties.keys():
        print(f"   - {prop_name}")
    
    print("\nPropriétés Test:")
    for prop_name in test_text_data._properties.keys():
        print(f"   - {prop_name}")
else:
    print("⚠️  Propriétés non encore calculées. Exécutez calculate_builtin_properties()")

📊 Propriétés textuelles disponibles:
--------------------------------------------------------------------------------

Propriétés Train:
   - Text Length
   - Average Word Length
   - Max Word Length
   - % Special Characters
   - % Punctuation
   - Language
   - Sentiment
   - Subjectivity
   - Average Words Per Sentence
   - Reading Ease
   - Lexical Density

Propriétés Test:
   - Text Length
   - Average Word Length
   - Max Word Length
   - % Special Characters
   - % Punctuation
   - Language
   - Sentiment
   - Subjectivity
   - Average Words Per Sentence
   - Reading Ease
   - Lexical Density


---

## 📊 Résumé et Recommandations

In [13]:
print("\n" + "="*80)
print("✅ NIVEAU 2 : DRIFT ET DISTRIBUTION - TERMINÉ")
print("="*80)

print("\n📂 Rapport généré:")
print(f"   {TESTING_DIR / 'deepchecks_nlp_drift_report.html'}")

print("\n💡 Interprétation des résultats:")
print("   ✅ Tous les checks réussis : Distributions similaires train/test")
print("   ⚠️  Label Drift détecté : Revoir le split ou rééquilibrer")
print("   ⚠️  Property Drift détecté : Analyser les différences de propriétés")
print("   ⚠️  Embeddings Drift : Dérive sémantique - revoir les données")
print("   ❌ Train Test Mix : Contamination détectée - refaire le split")

print("\n🔍 Métriques de drift:")
print("   • KL Divergence < 0.1 : Drift acceptable")
print("   • KL Divergence > 0.1 : Drift significatif")
print("   • Cosine Similarity > 0.95 : Embeddings similaires")

print("\n🔗 Ouvrez le rapport HTML pour visualiser les détails")
print("="*80)


✅ NIVEAU 2 : DRIFT ET DISTRIBUTION - TERMINÉ

📂 Rapport généré:
   e:\MLOps\mlops_election\testing\deepchecks_nlp_drift_report.html

💡 Interprétation des résultats:
   ✅ Tous les checks réussis : Distributions similaires train/test
   ⚠️  Label Drift détecté : Revoir le split ou rééquilibrer
   ⚠️  Property Drift détecté : Analyser les différences de propriétés
   ⚠️  Embeddings Drift : Dérive sémantique - revoir les données
   ❌ Train Test Mix : Contamination détectée - refaire le split

🔍 Métriques de drift:
   • KL Divergence < 0.1 : Drift acceptable
   • KL Divergence > 0.1 : Drift significatif
   • Cosine Similarity > 0.95 : Embeddings similaires

🔗 Ouvrez le rapport HTML pour visualiser les détails


---

## 📚 Documentation

### Checks exécutés

| Check | Description | Critère |
|-------|-------------|---------|
| **Label Drift** | Distribution des labels | KL divergence < 0.1 |
| **Property Drift** | Propriétés textuelles | Statistically insignificant |
| **Text Embeddings Drift** | Drift sémantique | Cosine similarity > 0.95 |
| **Train Test Samples Mix** | Contamination | 0 overlap |

### Propriétés textuelles calculées

- **Longueur du texte** (nombre de caractères)
- **Nombre de mots**
- **Taux de vocabulaire unique**
- **Sentiment lexical** (si disponible)
- **Complexité syntaxique**

### Actions si drift détecté

1. **Label Drift** :
   - Vérifier la stratification du split
   - Rééquilibrer les classes si nécessaire
   - Utiliser StratifiedShuffleSplit

2. **Property Drift** :
   - Analyser les distributions de longueur
   - Vérifier la cohérence du preprocessing
   - Normaliser les propriétés

3. **Embeddings Drift** :
   - Vérifier la cohérence sémantique
   - Analyser les topics train vs test
   - Considérer le domain adaptation

### Prochaines étapes

➡️ **NIVEAU 3** : Exécuter `deepchecks_performance.ipynb` pour évaluer le modèle

### Ressources
- [Deepchecks NLP Docs](https://docs.deepchecks.com/stable/nlp/index.html)
- [Train-Test Validation Suite](https://docs.deepchecks.com/stable/nlp/auto_checks/train_test_validation/index.html)